In [1]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from tabm import TabM
from tabicl import TabICLClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, average_precision_score, f1_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import precision_recall_curve
import sys
# Get the current working directory of the notebook
current_dir = os.getcwd()
# Get the parent directory
parent_dir = os.path.abspath(os.path.join(current_dir, os.pardir))
# Add the parent directory to sys.path
sys.path.insert(0, parent_dir)
from imputation import * 
from external_test import *
from summary_table import *
from feature_importance import *
# Show all rows
pd.set_option('display.max_rows', None)
# Show all columns
pd.set_option('display.max_columns', None)
# Do not truncate column values
pd.set_option('display.max_colwidth', None)

In [2]:
# Define input columns and target
X_columns = [
    #'hhidpn', 'NIWWAVE','demcls',
     'child', 'lbrf', 'shlt', 'ageym', 'height', 'weight', 'smokev', 'proxy', 'effort', 'hibpe', 'diabe', 'vgactx', 'slfmem', 'livpar', 'momage', 'dadage', 'livsib', 'hlthlm', 'hosp', 'nrshom', 'nrstim', 'nrsnit', 'doctim', 'depres', 'sleepr', 'whappy', 'flone', 'fsad', 'going', 'enlife', 'drink', 'smoken', 'cancre', 'lunge', 'hearte', 'stroke', 'arthre', 'toilta', 'adl5a', 'mapa', 'walksa', 'walk1a', 'sita', 'chaira', 'climsa', 'clim1a', 'stoopa', 'lifta', 'dimea', 'armsa', 'pusha', 'mobila', 'lgmusa', 'grossa', 'finea', 'gender', 'edstg2 8to11', 'edstg3 12', 'edstg4 13above', 'cendiv2 mid atlantic', 'cendiv3 en central', 'cendiv4 wn central', 'cendiv5 s atlantic', 'cendiv6 es central', 'cendiv7 ws central', 'cendiv8 mountain', 'cendiv9 pacific', 'cendiv11 not us or inc us terr', 'mstat2 married spouse absent', 'mstat3 partnered', 'mstat4 separated', 'mstat5 divorced', 'mstat7 widowed', 'mstat8 never married', 'raceeth1 ', 'raceeth2 ', 'raceeth3 '
]

y_column = 'y'

train_path = '../data/preprocessed data/2000(00-06y)encode.csv'

# Directories and file paths
grid_search_dir = f'../data/2000-2006'
test_path_2016 = f'../data/2000-2006/2010(10-16y)encode.csv'
test_path_2018 = f'../data/2000-2008/2010(10-18y)encode.csv'

In [3]:
train = pd.read_csv(train_path)
test_2016 = pd.read_csv(test_path_2016)
test_2018 = pd.read_csv(test_path_2018)

In [4]:
X_train, y_train = impute_data_and_y(train, 'mice',X_columns, y_column)

**2000-2006**

In [5]:
X_test, y_test = test_2016[X_columns], test_2016[y_column]
model = TabICLClassifier(n_estimators=16, batch_size=8, random_state=42)
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]
optimal_threshold = 0.1785
y_pred = (y_prob >= optimal_threshold).astype(int)

auc = roc_auc_score(y_test, y_prob)
acc = accuracy_score(y_test, y_pred)
aupr = average_precision_score(y_test, y_prob)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("TabICLClassifier with MICE Imputation — External Test (2010-2016)")
print(f"Optimal Threshold Used: {optimal_threshold:.4f}")
print(
    f"Final test metrics: "
    f"AUC = {auc:.4f}, ACC = {acc:.4f}, AUPR = {aupr:.4f}, "
    f"Precision = {precision:.4f}, Recall = {recall:.4f}, F1 = {f1:.4f}"
)

TabICLClassifier with MICE Imputation — External Test (2010-2016)
Optimal Threshold Used: 0.1785
Final test metrics: AUC = 0.8290, ACC = 0.8930, AUPR = 0.3325, Precision = 0.3638, Recall = 0.3886, F1 = 0.3758


**2000-2008**

In [6]:
X_test, y_test = test_2018[X_columns], test_2018[y_column]
model = TabICLClassifier(n_estimators=16, batch_size=8, random_state=42)
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]
optimal_threshold = 0.1785
y_pred = (y_prob >= optimal_threshold).astype(int)

auc = roc_auc_score(y_test, y_prob)
acc = accuracy_score(y_test, y_pred)
aupr = average_precision_score(y_test, y_prob)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("TabICLClassifier with MICE Imputation — External Test (2010-2018)")
print(f"Optimal Threshold Used: {optimal_threshold:.4f}")
print(
    f"Final test metrics: "
    f"AUC = {auc:.4f}, ACC = {acc:.4f}, AUPR = {aupr:.4f}, "
    f"Precision = {precision:.4f}, Recall = {recall:.4f}, F1 = {f1:.4f}"
)

TabICLClassifier with MICE Imputation — External Test (2010-2018)
Optimal Threshold Used: 0.1785
Final test metrics: AUC = 0.8256, ACC = 0.8858, AUPR = 0.3565, Precision = 0.3969, Recall = 0.3663, F1 = 0.3810
